In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ======================
# データ読み込み
# ======================
df = pd.read_csv("/kaggle/input/datasets/sakoshota/dataset/ETTh1.csv")

df["date"] = pd.to_datetime(df["date"])

print("="*50)
print("データサイズ")
print(df.shape)

print("\n列名")
print(df.columns.tolist())

# ======================
# 欠損値確認
# ======================
print("\n"+"="*50)
print("欠損値確認")

print(df.isnull().sum())

# ======================
# 基本統計量
# ======================
print("\n"+"="*50)
print("基本統計量")

print(df.describe())

# ======================
# OT時系列可視化
# ======================
plt.figure(figsize=(14,5))
plt.plot(df["date"], df["OT"])
plt.title("Oil Temperature (OT)")
plt.xlabel("Date")
plt.ylabel("OT")
plt.tight_layout()
plt.show()

# ======================
# 相関分析
# ======================
plt.figure(figsize=(8,6))

corr = df.drop(columns=["date"]).corr()

sns.heatmap(
    corr,
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()

# ======================
# OT自己相関
# ======================
from pandas.plotting import autocorrelation_plot

plt.figure(figsize=(10,5))
autocorrelation_plot(df["OT"])
plt.title("Autocorrelation of OT")
plt.show()

# ======================
# 時間特徴量
# ======================
df["hour"] = df["date"].dt.hour
df["dayofweek"] = df["date"].dt.dayofweek

# ======================
# 日周期
# ======================
hour_mean = df.groupby("hour")["OT"].mean()

plt.figure(figsize=(8,5))
hour_mean.plot(marker="o")
plt.title("Average OT by Hour")
plt.xlabel("Hour")
plt.ylabel("Average OT")
plt.grid()
plt.show()

# ======================
# 週周期
# ======================
week_mean = df.groupby("dayofweek")["OT"].mean()

plt.figure(figsize=(8,5))
week_mean.plot(marker="o")
plt.title("Average OT by Day of Week")
plt.xlabel("Day of Week")
plt.ylabel("Average OT")
plt.grid()
plt.show()

In [ ]:
lags = range(0, 49)

features = [
    "HUFL",
    "HULL",
    "MUFL",
    "MULL",
    "LUFL",
    "LULL"
]

plt.figure(figsize=(12, 6))

for feature in features:

    corrs = []

    for lag in lags:

        corr = df["OT"].corr(df[feature].shift(lag))
        corrs.append(corr)

    plt.plot(lags, corrs, label=feature)

plt.xlabel("Lag (hours)")
plt.ylabel("Correlation with OT")
plt.title("Lag Correlation Analysis")
plt.legend()
plt.grid()
plt.show()